# PHẦN IV — KHỞI TẠO CONTROLLER VM

## Bản chi tiết đã chỉnh theo thực tế lab

File này giữ cấu trúc chi tiết như bản `phan_4_khoi_tao_controller_vm_detailed.ipynb` cũ, nhưng đã chỉnh lại theo thực tế bạn kiểm tra trong console:

- NIC trong VM là `enp1s0` và `enp2s0`, không phải `ens3/ens4`.
- Lab IP của Controller VM là `192.168.150.21`, `192.168.150.22`, `192.168.150.23`.
- Gateway lab thực tế là `192.168.150.254`.
- Management route đi qua gateway tương ứng trên host:
  - Controller 1: `10.0.0.0/8 via 10.0.0.21`
  - Controller 2: `10.0.0.0/8 via 10.0.0.22`
  - Controller 3: `10.0.0.0/8 via 10.0.0.23`

## Mục tiêu phần này

Phần này dùng template đã tạo ở **PHẦN III** để khởi tạo 3 Controller VM, mỗi VM nằm trên một server vật lý khác nhau.

Mỗi Controller VM có 2 NIC:

| NIC trong VM | Bridge trên host | Mục đích |
|---|---|---|
| `enp1s0` | `br-mgmt` | Management network nội bộ OpenStack |
| `enp2s0` | `br-lab` | Lab/uplink network để SSH, Internet, tải package |

## IP plan Controller VM thực tế

| VM | Chạy trên Server | Hostname | Management IP | Management Route | Lab IP | Default Gateway |
|---|---|---|---|---|---|---|
| Controller VM1 | Server A `.150.9` | `controller-1` | `10.0.0.11/24` | `10.0.0.0/8 via 10.0.0.21` | `192.168.150.21/24` | `192.168.150.254` |
| Controller VM2 | Server B `.150.10` | `controller-2` | `10.0.0.12/24` | `10.0.0.0/8 via 10.0.0.22` | `192.168.150.22/24` | `192.168.150.254` |
| Controller VM3 | Server C `.150.11` | `controller-3` | `10.0.0.13/24` | `10.0.0.0/8 via 10.0.0.23` | `192.168.150.23/24` | `192.168.150.254` |

## Điều kiện trước khi làm phần này

Trên mỗi server vật lý phải đã có:

- `br-mgmt` hoạt động.
- `br-lab` hoạt động.
- LVM thin pool `vg-lab/thin-pool`.
- Template image:
  ```bash
  /var/lib/libvirt/templates/tpl-controller.img
  ```
- Package `cloud-image-utils` để có lệnh `cloud-localds`.
- Package `virtinst` để có lệnh `virt-install`.

## Lưu ý quan trọng

- Không boot trực tiếp file template.
- Mỗi VM phải có disk riêng trong `/dev/vg-lab/`.
- NIC thứ nhất trong `virt-install` gắn `br-mgmt` → thường vào VM thành `enp1s0`.
- NIC thứ hai trong `virt-install` gắn `br-lab` → thường vào VM thành `enp2s0`.
- Nếu thứ tự NIC bị đảo, cần kiểm tra lại bằng `ip addr` trong VM.
- `enp2s0` là NIC có default route ra ngoài qua `192.168.150.254`.
- `enp1s0` là management NIC, chỉ dùng route nội bộ `10.0.0.0/8`.


## BƯỚC 17 — Tạo cloud-init datasource cho từng Controller VM

Cloud-init gồm 2 file:

1. `user-data`: cấu hình hostname, password, SSH, network.
2. `meta-data`: định danh instance.

Sau đó đóng gói thành seed ISO bằng:

```bash
cloud-localds <seed.iso> <user-data> <meta-data>
```

Seed ISO sẽ được gắn vào VM như CD-ROM khi boot lần đầu.

## Cấu trúc network trong cloud-init

Trong VM thực tế:

- `enp1s0`: nối vào `br-mgmt`, đặt IP `10.0.0.x`.
- `enp2s0`: nối vào `br-lab`, đặt IP `192.168.150.x`.

`enp2s0` có default route, vì đây là đường ra ngoài lab/Internet.

`enp1s0` có route nội bộ tới `10.0.0.0/8`, vì đây là management network dùng cho OpenStack.


## BƯỚC 17.1 — Cloud-init cho Controller VM1 trên Server A

Chạy trên **Server A**.

Thông tin VM1:

- VM name: `controller-vm-1`
- Hostname: `controller-1`
- FQDN: `controller-1.openstack.lab`
- Management NIC: `enp1s0`
- Management IP: `10.0.0.11/24`
- Management route: `10.0.0.0/8 via 10.0.0.21`
- Lab NIC: `enp2s0`
- Lab IP: `192.168.150.21/24`
- Default gateway: `192.168.150.254`


In [ ]:
# ============================================================
# BƯỚC 17.1 — SERVER A: tạo cloud-init cho Controller VM1
# ============================================================

# Tạo user-data cho Controller VM1
# enp1s0: management network qua br-mgmt
# enp2s0: lab/uplink network qua br-lab
#
# Lưu ý:
# - enp1s0 KHÔNG đặt default route.
# - enp2s0 đặt default route qua 192.168.150.254.
# - Route 10.0.0.0/8 đi qua gateway management 10.0.0.21.
cat > /tmp/ctrl1-user-data << EOF
#cloud-config
hostname: controller-1
fqdn: controller-1.openstack.lab
manage_etc_hosts: true

# Password login cho user mặc định ubuntu
password: 123.abc
chpasswd:
  expire: false
ssh_pwauth: true

# Cấu hình network bên trong VM
network:
  version: 2
  ethernets:
    enp1s0:
      dhcp4: false
      dhcp6: false
      addresses:
        - "10.0.0.11/24"
      routes:
        - to: "10.0.0.0/8"
          via: "10.0.0.21"

    enp2s0:
      dhcp4: false
      dhcp6: false
      addresses:
        - "192.168.150.21/24"
      routes:
        - to: default
          via: "192.168.150.254"
      nameservers:
        addresses: [8.8.8.8, 1.1.1.1]

# Lệnh chạy sau boot lần đầu
runcmd:
  - ip link set enp1s0 mtu 1450 || true
  - systemctl restart ssh || true
EOF

# Tạo meta-data cho Controller VM1
cat > /tmp/ctrl1-meta-data << EOF
instance-id: controller-1
local-hostname: controller-1
EOF

# Đóng gói user-data + meta-data thành seed ISO
cloud-localds /tmp/ctrl1-seed.iso \
  /tmp/ctrl1-user-data \
  /tmp/ctrl1-meta-data

# Verify seed ISO
ls -lh /tmp/ctrl1-seed.iso


## BƯỚC 17.2 — Cloud-init cho Controller VM2 trên Server B

Chạy trên **Server B**.

Thông tin VM2:

- VM name: `controller-vm-2`
- Hostname: `controller-2`
- FQDN: `controller-2.openstack.lab`
- Management NIC: `enp1s0`
- Management IP: `10.0.0.12/24`
- Management route: `10.0.0.0/8 via 10.0.0.22`
- Lab NIC: `enp2s0`
- Lab IP: `192.168.150.22/24`
- Default gateway: `192.168.150.254`


In [ ]:
# ============================================================
# BƯỚC 17.2 — SERVER B: tạo cloud-init cho Controller VM2
# ============================================================

cat > /tmp/ctrl2-user-data << EOF
#cloud-config
hostname: controller-2
fqdn: controller-2.openstack.lab
manage_etc_hosts: true

password: 123.abc
chpasswd:
  expire: false
ssh_pwauth: true

network:
  version: 2
  ethernets:
    enp1s0:
      dhcp4: false
      dhcp6: false
      addresses:
        - "10.0.0.12/24"
      routes:
        - to: "10.0.0.0/8"
          via: "10.0.0.22"

    enp2s0:
      dhcp4: false
      dhcp6: false
      addresses:
        - "192.168.150.22/24"
      routes:
        - to: default
          via: "192.168.150.254"
      nameservers:
        addresses: [8.8.8.8, 1.1.1.1]

runcmd:
  - ip link set enp1s0 mtu 1450 || true
  - systemctl restart ssh || true
EOF

cat > /tmp/ctrl2-meta-data << EOF
instance-id: controller-2
local-hostname: controller-2
EOF

cloud-localds /tmp/ctrl2-seed.iso \
  /tmp/ctrl2-user-data \
  /tmp/ctrl2-meta-data

ls -lh /tmp/ctrl2-seed.iso


## BƯỚC 17.3 — Cloud-init cho Controller VM3 trên Server C

Chạy trên **Server C**.

Thông tin VM3:

- VM name: `controller-vm-3`
- Hostname: `controller-3`
- FQDN: `controller-3.openstack.lab`
- Management NIC: `enp1s0`
- Management IP: `10.0.0.13/24`
- Management route: `10.0.0.0/8 via 10.0.0.23`
- Lab NIC: `enp2s0`
- Lab IP: `192.168.150.23/24`
- Default gateway: `192.168.150.254`


In [ ]:
# ============================================================
# BƯỚC 17.3 — SERVER C: tạo cloud-init cho Controller VM3
# ============================================================

cat > /tmp/ctrl3-user-data << EOF
#cloud-config
hostname: controller-3
fqdn: controller-3.openstack.lab
manage_etc_hosts: true

password: 123.abc
chpasswd:
  expire: false
ssh_pwauth: true

network:
  version: 2
  ethernets:
    enp1s0:
      dhcp4: false
      dhcp6: false
      addresses:
        - "10.0.0.13/24"
      routes:
        - to: "10.0.0.0/8"
          via: "10.0.0.23"

    enp2s0:
      dhcp4: false
      dhcp6: false
      addresses:
        - "192.168.150.23/24"
      routes:
        - to: default
          via: "192.168.150.254"
      nameservers:
        addresses: [8.8.8.8, 1.1.1.1]

runcmd:
  - ip link set enp1s0 mtu 1450 || true
  - systemctl restart ssh || true
EOF

cat > /tmp/ctrl3-meta-data << EOF
instance-id: controller-3
local-hostname: controller-3
EOF

cloud-localds /tmp/ctrl3-seed.iso \
  /tmp/ctrl3-user-data \
  /tmp/ctrl3-meta-data

ls -lh /tmp/ctrl3-seed.iso


## BƯỚC 17.4 — Verify cloud-init datasource

Chạy trên từng server tương ứng để kiểm tra seed ISO đã tồn tại.

Nếu file ISO không tồn tại hoặc dung lượng bằng 0, VM có thể boot nhưng không nhận hostname/IP/password.

## Ghi chú kiểm tra nhanh YAML

Nếu cloud-init network không chạy, lỗi thường do YAML sai indent. Cần kiểm tra kỹ:

- `network:` nằm ở cấp root.
- `ethernets:` thụt vào dưới `network`.
- `enp1s0`, `enp2s0` thụt vào dưới `ethernets`.
- `routes:` là list, mỗi route bắt đầu bằng `-`.


In [ ]:
# ============================================================
# VERIFY cloud-init seed ISO
# Chạy trên đúng server tương ứng
# ============================================================

# Server A
ls -lh /tmp/ctrl1-seed.iso /tmp/ctrl1-user-data /tmp/ctrl1-meta-data 2>/dev/null || true

# Server B
ls -lh /tmp/ctrl2-seed.iso /tmp/ctrl2-user-data /tmp/ctrl2-meta-data 2>/dev/null || true

# Server C
ls -lh /tmp/ctrl3-seed.iso /tmp/ctrl3-user-data /tmp/ctrl3-meta-data 2>/dev/null || true

# Kiểm tra package cloud-localds
which cloud-localds

# Xem lại nội dung user-data nếu cần
# cat /tmp/ctrl1-user-data
# cat /tmp/ctrl2-user-data
# cat /tmp/ctrl3-user-data


## BƯỚC 18 — Tạo disk VM từ template và khởi động

Mỗi Controller VM cần một disk riêng được tạo từ template:

```bash
/var/lib/libvirt/templates/tpl-controller.img
```

Disk VM sẽ nằm trong LVM:

```bash
/dev/vg-lab/ctrl-vm-X
```

## Cách làm

1. Tạo thin volume trong LVM.
2. Copy nội dung template vào volume bằng `dd`.
3. Tạo VM bằng `virt-install`.
4. Gắn 2 NIC:
   - NIC1: `br-mgmt` → trong VM thường là `enp1s0`
   - NIC2: `br-lab` → trong VM thường là `enp2s0`
5. Gắn seed ISO cloud-init.
6. Boot VM bằng `--import`.

## Lưu ý

Lệnh trong tài liệu thầy có đoạn:

```bash
sudo cp tpl-controller.img /dev/vg-lab/ctrl-vm-1 2>/dev/null || sudo lvcreate ...
```

Trong thực tế nên làm rõ hơn:

- Tạo LV trước.
- Sau đó `dd` template vào LV.

Như vậy dễ kiểm soát và ít nhầm hơn.


## BƯỚC 18.1 — Server A tạo Controller VM1

Chạy trên **Server A**.

Thông số VM1:

- VM name: `controller-vm-1`
- Disk: `/dev/vg-lab/ctrl-vm-1`
- RAM: `10240 MB`
- vCPU: `2`
- NIC1: `br-mgmt`
- NIC2: `br-lab`
- Seed ISO: `/tmp/ctrl1-seed.iso`


In [ ]:
# ============================================================
# BƯỚC 18.1 — SERVER A: tạo disk và boot Controller VM1
# ============================================================

# Di chuyển về thư mục template
cd /var/lib/libvirt/templates

# Kiểm tra template tồn tại trước khi tạo VM
ls -lh tpl-controller.img
qemu-img info tpl-controller.img | egrep "file format|virtual size"

# Kiểm tra thin pool tồn tại
sudo lvs | egrep "thin-pool|ctrl-vm-1" || true

# Tạo thin logical volume 100G cho Controller VM1
# Nếu LV đã tồn tại, lệnh này sẽ báo lỗi; khi đó cần kiểm tra bằng sudo lvs.
sudo lvcreate -V 100G --thin -n ctrl-vm-1 vg-lab/thin-pool

# Copy template vào LV của VM1
# Dùng bs=4M để tăng tốc ghi.
sudo dd if=/var/lib/libvirt/templates/tpl-controller.img \
  of=/dev/vg-lab/ctrl-vm-1 bs=4M status=progress

# Verify LV đã có
sudo lvs | grep ctrl-vm-1

# Tạo Controller VM1
# Thứ tự NIC rất quan trọng:
# NIC1 bridge=br-mgmt → thường là enp1s0 trong VM
# NIC2 bridge=br-lab  → thường là enp2s0 trong VM
virt-install \
  --name controller-vm-1 \
  --memory 10240 \
  --vcpus 2 \
  --disk /dev/vg-lab/ctrl-vm-1,format=raw,bus=virtio \
  --disk /tmp/ctrl1-seed.iso,device=cdrom \
  --os-variant ubuntu24.04 \
  --network bridge=br-mgmt,model=virtio \
  --network bridge=br-lab,model=virtio \
  --graphics vnc,listen=0.0.0.0 \
  --noautoconsole \
  --import

# Kiểm tra VM đã được tạo
virsh list --all


## BƯỚC 18.2 — Server B tạo Controller VM2

Chạy trên **Server B**.

Thông số VM2:

- VM name: `controller-vm-2`
- Disk: `/dev/vg-lab/ctrl-vm-2`
- RAM: `10240 MB`
- vCPU: `2`
- NIC1: `br-mgmt`
- NIC2: `br-lab`
- Seed ISO: `/tmp/ctrl2-seed.iso`


In [ ]:
# ============================================================
# BƯỚC 18.2 — SERVER B: tạo disk và boot Controller VM2
# ============================================================

cd /var/lib/libvirt/templates

# Kiểm tra template
ls -lh tpl-controller.img
qemu-img info tpl-controller.img | egrep "file format|virtual size"

# Kiểm tra thin pool
sudo lvs | egrep "thin-pool|ctrl-vm-2" || true

# Tạo thin logical volume cho VM2
sudo lvcreate -V 100G --thin -n ctrl-vm-2 vg-lab/thin-pool

# Copy template vào LV của VM2
sudo dd if=/var/lib/libvirt/templates/tpl-controller.img \
  of=/dev/vg-lab/ctrl-vm-2 bs=4M status=progress

# Verify LV
sudo lvs | grep ctrl-vm-2

# Tạo Controller VM2
virt-install \
  --name controller-vm-2 \
  --memory 10240 \
  --vcpus 2 \
  --disk /dev/vg-lab/ctrl-vm-2,format=raw,bus=virtio \
  --disk /tmp/ctrl2-seed.iso,device=cdrom \
  --os-variant ubuntu24.04 \
  --network bridge=br-mgmt,model=virtio \
  --network bridge=br-lab,model=virtio \
  --graphics vnc,listen=0.0.0.0 \
  --noautoconsole \
  --import

virsh list --all


## BƯỚC 18.3 — Server C tạo Controller VM3

Chạy trên **Server C**.

Thông số VM3:

- VM name: `controller-vm-3`
- Disk: `/dev/vg-lab/ctrl-vm-3`
- RAM: `10240 MB`
- vCPU: `2`
- NIC1: `br-mgmt`
- NIC2: `br-lab`
- Seed ISO: `/tmp/ctrl3-seed.iso`


In [ ]:
# ============================================================
# BƯỚC 18.3 — SERVER C: tạo disk và boot Controller VM3
# ============================================================

cd /var/lib/libvirt/templates

# Kiểm tra template
ls -lh tpl-controller.img
qemu-img info tpl-controller.img | egrep "file format|virtual size"

# Kiểm tra thin pool
sudo lvs | egrep "thin-pool|ctrl-vm-3" || true

# Tạo thin logical volume cho VM3
sudo lvcreate -V 100G --thin -n ctrl-vm-3 vg-lab/thin-pool

# Copy template vào LV của VM3
sudo dd if=/var/lib/libvirt/templates/tpl-controller.img \
  of=/dev/vg-lab/ctrl-vm-3 bs=4M status=progress

# Verify LV
sudo lvs | grep ctrl-vm-3

# Tạo Controller VM3
virt-install \
  --name controller-vm-3 \
  --memory 10240 \
  --vcpus 2 \
  --disk /dev/vg-lab/ctrl-vm-3,format=raw,bus=virtio \
  --disk /tmp/ctrl3-seed.iso,device=cdrom \
  --os-variant ubuntu24.04 \
  --network bridge=br-mgmt,model=virtio \
  --network bridge=br-lab,model=virtio \
  --graphics vnc,listen=0.0.0.0 \
  --noautoconsole \
  --import

virsh list --all


## BƯỚC 18.4 — Theo dõi boot VM

Sau khi `virt-install` hoàn tất, VM sẽ boot từ disk đã tạo.

Dùng `virsh list --all` để xem trạng thái VM.

Dùng `virsh console` để vào console nếu cần debug.

Thoát console bằng:

```bash
Ctrl + ]
```

## Lưu ý lỗi `stream had I/O failure`

Nếu console báo lỗi kiểu:

```text
error: stream had I/O failure
```

thì có thể VM chưa sẵn sàng console, hoặc serial console chưa được enable đúng trong image. Khi đó kiểm tra bằng:

```bash
virsh list --all
virsh dominfo <vm-name>
virsh vncdisplay <vm-name>
```

hoặc SSH trực tiếp vào IP đã cấu hình.


In [ ]:
# ============================================================
# Theo dõi VM boot
# Chạy trên server đang chứa VM tương ứng
# ============================================================

# Liệt kê VM
virsh list --all

# Console VM1 trên Server A
virsh console controller-vm-1

# Console VM2 trên Server B
# virsh console controller-vm-2

# Console VM3 trên Server C
# virsh console controller-vm-3

# Thoát console bằng Ctrl + ]


## BƯỚC 18.5 — Kiểm tra IP của Controller VM

Sau khi boot khoảng 60–120 giây, kiểm tra IP bằng `virsh domifaddr`.

Lưu ý:

- `domifaddr` đôi khi không hiện IP nếu QEMU guest agent chưa chạy.
- Nếu không hiện IP, SSH trực tiếp theo IP cloud-init đã cấu hình.
- Nếu cloud-init sai tên card, VM có thể không nhận IP. Khi đó vào console sửa netplan trực tiếp.


In [ ]:
# ============================================================
# Kiểm tra IP VM từ host
# ============================================================

# Server A
virsh domifaddr controller-vm-1

# Server B
virsh domifaddr controller-vm-2

# Server C
virsh domifaddr controller-vm-3

# Nếu không có output:
# 1. Đợi cloud-init chạy xong
# 2. Vào console kiểm tra ip addr
# 3. SSH trực tiếp theo IP đã gán


## BƯỚC 18.6 — SSH vào Controller VM

Có thể SSH bằng management IP hoặc lab IP.

Management IP dùng để các controller nói chuyện nội bộ với nhau.

Lab IP dùng để SSH từ máy ngoài / truy cập Internet / tải package.

## IP SSH thực tế

- Controller 1:
  - `ssh ubuntu@10.0.0.11`
  - `ssh ubuntu@192.168.150.21`
- Controller 2:
  - `ssh ubuntu@10.0.0.12`
  - `ssh ubuntu@192.168.150.22`
- Controller 3:
  - `ssh ubuntu@10.0.0.13`
  - `ssh ubuntu@192.168.150.23`


In [ ]:
# ============================================================
# SSH vào Controller VM
# ============================================================

# Controller VM1
ssh ubuntu@10.0.0.11
ssh ubuntu@192.168.150.21

# Controller VM2
ssh ubuntu@10.0.0.12
ssh ubuntu@192.168.150.22

# Controller VM3
ssh ubuntu@10.0.0.13
ssh ubuntu@192.168.150.23

# Password mặc định trong lab:
# 123.abc


## BƯỚC 18.7 — Verify bên trong Controller VM

Sau khi SSH vào từng Controller VM, chạy các lệnh dưới đây.

Mục tiêu kiểm tra:

- Hostname đúng.
- NIC `enp1s0` có IP management.
- NIC `enp2s0` có IP lab.
- Default route đi qua `192.168.150.254`.
- Route `10.0.0.0/8` đi qua gateway management tương ứng.
- Docker đã cài.
- Kolla-Ansible đã có trong venv.
- VM ra Internet được.


In [ ]:
# ============================================================
# Verify bên trong Controller VM
# Chạy sau khi SSH vào VM
# ============================================================

# Kiểm tra hostname
hostname
hostname -f

# Kiểm tra IP management
# Controller 1 kỳ vọng: 10.0.0.11/24
# Controller 2 kỳ vọng: 10.0.0.12/24
# Controller 3 kỳ vọng: 10.0.0.13/24
ip addr show enp1s0

# Kiểm tra IP lab/uplink
# Controller 1 kỳ vọng: 192.168.150.21/24
# Controller 2 kỳ vọng: 192.168.150.22/24
# Controller 3 kỳ vọng: 192.168.150.23/24
ip addr show enp2s0

# Kiểm tra default route và route management
ip route

# Kỳ vọng:
# default via 192.168.150.254 dev enp2s0
# 10.0.0.0/8 via 10.0.0.21/22/23 dev enp1s0

# Kiểm tra Docker
docker --version
sudo systemctl status docker --no-pager

# Kiểm tra Kolla-Ansible
ls -lh /opt/kolla-venv/bin/kolla-ansible
/opt/kolla-venv/bin/kolla-ansible --version

# Kiểm tra Python venv
/opt/kolla-venv/bin/python --version
/opt/kolla-venv/bin/pip list | egrep "kolla|ansible|docker"

# Kiểm tra Internet
ping -c 2 192.168.150.254
ping -c 2 8.8.8.8
ping -c 2 google.com

# Kiểm tra time sync
timedatectl
chronyc tracking || true


## BƯỚC 18.8 — Kiểm tra kết nối giữa 3 Controller qua `br-mgmt`

Sau khi tạo đủ 3 Controller VM, SSH vào Controller VM1 và ping sang VM2/VM3 bằng IP management.

Chạy trên `controller-1`.

Mục tiêu:

- `controller-1` ping được `controller-2` qua `10.0.0.12`.
- `controller-1` ping được `controller-3` qua `10.0.0.13`.
- Đây là điều kiện quan trọng trước khi triển khai Kolla-Ansible multi-controller.


In [ ]:
# ============================================================
# Chạy trong controller-1
# Kiểm tra management network giữa 3 controller
# ============================================================

# Ping controller-2 management IP
ping -c 3 10.0.0.12

# Ping controller-3 management IP
ping -c 3 10.0.0.13

# Kiểm tra SSH nội bộ nếu cần
ssh ubuntu@10.0.0.12 hostname
ssh ubuntu@10.0.0.13 hostname


## BƯỚC 18.9 — Kiểm tra kết nối Internet qua `enp2s0`

Chạy trong từng Controller VM.

Nếu ping IP được nhưng ping domain không được, lỗi thường là DNS.

Nếu không ping được `8.8.8.8`, lỗi thường là route/gateway/uplink.

Gateway lab trong môi trường thực tế là:

```bash
192.168.150.254
```


In [ ]:
# ============================================================
# Chạy trong từng Controller VM
# ============================================================

# Kiểm tra default route
ip route | grep default

# Ping gateway lab
ping -c 3 192.168.150.254

# Ping Internet bằng IP
ping -c 3 8.8.8.8

# Ping Internet bằng domain
ping -c 3 google.com

# Kiểm tra DNS resolver
resolvectl status || cat /etc/resolv.conf


## BƯỚC 18.10 — Sửa nhanh nếu VM đã tạo sai IP hoặc sai tên card

Trường hợp bạn đã tạo VM với cloud-init cũ hoặc IP cũ (`192.168.150.91/101/111`), có thể sửa trực tiếp trong VM.

Ví dụ trong ảnh thực tế:

- Card là `enp1s0`, `enp2s0`.
- Controller 2 dùng:
  - `10.0.0.12/24`
  - `192.168.150.22/24`
  - default via `192.168.150.254`.

File thường là:

```bash
/etc/netplan/50-cloud-init.yaml
```

Sau khi sửa:

```bash
sudo netplan generate
sudo netplan apply
```

Nếu cần đổi hostname:

```bash
sudo hostnamectl set-hostname controller-2
```


In [ ]:
# ============================================================
# Ví dụ sửa netplan trực tiếp cho controller-2
# Dựa trên ảnh thực tế bạn gửi
# ============================================================

sudo tee /etc/netplan/50-cloud-init.yaml << EOF
network:
  version: 2
  ethernets:
    enp1s0:
      dhcp4: false
      addresses:
        - "10.0.0.12/24"
      routes:
        - to: "10.0.0.0/8"
          via: "10.0.0.22"
      dhcp6: false

    enp2s0:
      dhcp4: false
      addresses:
        - "192.168.150.22/24"
      routes:
        - to: default
          via: "192.168.150.254"
      nameservers:
        addresses: [8.8.8.8, 1.1.1.1]
EOF

sudo netplan generate
sudo netplan apply

ip addr
ip route


## BƯỚC 18.11 — Mẫu sửa netplan cho cả 3 Controller nếu cần

Dùng phần này khi VM đã tạo rồi nhưng cloud-init không áp dụng đúng.

### Controller 1

- Hostname: `controller-1`
- `enp1s0`: `10.0.0.11/24`
- route: `10.0.0.0/8 via 10.0.0.21`
- `enp2s0`: `192.168.150.21/24`
- default: `192.168.150.254`

### Controller 2

- Hostname: `controller-2`
- `enp1s0`: `10.0.0.12/24`
- route: `10.0.0.0/8 via 10.0.0.22`
- `enp2s0`: `192.168.150.22/24`
- default: `192.168.150.254`

### Controller 3

- Hostname: `controller-3`
- `enp1s0`: `10.0.0.13/24`
- route: `10.0.0.0/8 via 10.0.0.23`
- `enp2s0`: `192.168.150.23/24`
- default: `192.168.150.254`


In [ ]:
# ============================================================
# Controller 1 — sửa hostname và netplan nếu cần
# ============================================================

sudo hostnamectl set-hostname controller-1

sudo tee /etc/netplan/50-cloud-init.yaml << EOF
network:
  version: 2
  ethernets:
    enp1s0:
      dhcp4: false
      dhcp6: false
      addresses:
        - "10.0.0.11/24"
      routes:
        - to: "10.0.0.0/8"
          via: "10.0.0.21"

    enp2s0:
      dhcp4: false
      dhcp6: false
      addresses:
        - "192.168.150.21/24"
      routes:
        - to: default
          via: "192.168.150.254"
      nameservers:
        addresses: [8.8.8.8, 1.1.1.1]
EOF

sudo netplan generate
sudo netplan apply

hostname
ip addr
ip route


In [ ]:
# ============================================================
# Controller 2 — sửa hostname và netplan nếu cần
# ============================================================

sudo hostnamectl set-hostname controller-2

sudo tee /etc/netplan/50-cloud-init.yaml << EOF
network:
  version: 2
  ethernets:
    enp1s0:
      dhcp4: false
      dhcp6: false
      addresses:
        - "10.0.0.12/24"
      routes:
        - to: "10.0.0.0/8"
          via: "10.0.0.22"

    enp2s0:
      dhcp4: false
      dhcp6: false
      addresses:
        - "192.168.150.22/24"
      routes:
        - to: default
          via: "192.168.150.254"
      nameservers:
        addresses: [8.8.8.8, 1.1.1.1]
EOF

sudo netplan generate
sudo netplan apply

hostname
ip addr
ip route


In [ ]:
# ============================================================
# Controller 3 — sửa hostname và netplan nếu cần
# ============================================================

sudo hostnamectl set-hostname controller-3

sudo tee /etc/netplan/50-cloud-init.yaml << EOF
network:
  version: 2
  ethernets:
    enp1s0:
      dhcp4: false
      dhcp6: false
      addresses:
        - "10.0.0.13/24"
      routes:
        - to: "10.0.0.0/8"
          via: "10.0.0.23"

    enp2s0:
      dhcp4: false
      dhcp6: false
      addresses:
        - "192.168.150.23/24"
      routes:
        - to: default
          via: "192.168.150.254"
      nameservers:
        addresses: [8.8.8.8, 1.1.1.1]
EOF

sudo netplan generate
sudo netplan apply

hostname
ip addr
ip route


## Debug nhanh lỗi thường gặp

### 1. VM không nhận IP cloud-init

Kiểm tra trong VM console:

```bash
cloud-init status --long
cat /var/log/cloud-init.log
cat /var/log/cloud-init-output.log
ip addr
```

Nguyên nhân thường gặp:

- Seed ISO không gắn vào VM.
- File `user-data` sai format YAML.
- Tên NIC trong cloud-init không đúng (`enp1s0`, `enp2s0` bị đảo hoặc khác).
- Cloud-init đã chạy một lần với metadata cũ.

### 2. SSH không vào được

Kiểm tra trong VM:

```bash
sudo systemctl status ssh
sudo grep PasswordAuthentication /etc/ssh/sshd_config
ip addr
```

Từ host kiểm tra:

```bash
ping <VM_IP>
nc -vz <VM_IP> 22
```

### 3. VM không ra Internet

Kiểm tra trong VM:

```bash
ip route
ping -c 3 192.168.150.254
ping -c 3 8.8.8.8
ping -c 3 google.com
```

Kiểm tra trên host:

```bash
ip addr show br-lab
bridge link
```

### 4. `virsh domifaddr` không hiện IP

Không nhất thiết là lỗi. Có thể VM chưa cài/chưa chạy QEMU guest agent.

Dùng console hoặc SSH trực tiếp theo IP đã gán.

### 5. Tạo LV bị lỗi vì đã tồn tại

Kiểm tra:

```bash
sudo lvs
```

Nếu muốn xóa VM và tạo lại:

```bash
virsh destroy controller-vm-1
virsh undefine controller-vm-1 --nvram
sudo lvremove -y /dev/vg-lab/ctrl-vm-1
```

Cẩn thận chỉ xóa đúng VM cần làm lại.

### 6. Console báo `stream had I/O failure`

Có thể do serial console chưa sẵn sàng hoặc image chưa enable serial console.

Kiểm tra VM vẫn đang chạy không:

```bash
virsh list --all
virsh dominfo controller-vm-1
virsh vncdisplay controller-vm-1
```

Nếu SSH được thì không cần dùng console.


## Checklist hoàn thành PHẦN IV

Trước khi chuyển sang phần Kolla-Ansible / triển khai OpenStack, kiểm tra đủ:

- [ ] Controller VM1 đã boot trên Server A.
- [ ] Controller VM2 đã boot trên Server B.
- [ ] Controller VM3 đã boot trên Server C.
- [ ] VM1 có IP `10.0.0.11` và `192.168.150.21`.
- [ ] VM2 có IP `10.0.0.12` và `192.168.150.22`.
- [ ] VM3 có IP `10.0.0.13` và `192.168.150.23`.
- [ ] `enp1s0` là management NIC.
- [ ] `enp2s0` là lab/uplink NIC.
- [ ] Default route đi qua `192.168.150.254`.
- [ ] Route `10.0.0.0/8` đi qua gateway management tương ứng.
- [ ] SSH vào được cả 3 VM.
- [ ] Docker chạy trong cả 3 VM.
- [ ] `/opt/kolla-venv/bin/kolla-ansible` tồn tại.
- [ ] Controller VM1 ping được VM2/VM3 qua `10.0.0.x`.
- [ ] Cả 3 VM ra Internet qua `enp2s0`.
- [ ] Time sync/chrony hoạt động.
